# Day 2 · Notebook 01 — Transfer Learning with MobileNetV2 on CIFAR-10

**Goal:** take a network that was trained on 1.28 M ImageNet photos and *re-purpose* it for a new
10-class problem (CIFAR-10) in a few minutes on a laptop CPU.

By the end of this notebook you will have:

1. loaded an ImageNet-pretrained `torchvision` **MobileNetV2** and replaced its classification head,
2. compared **feature extraction** (frozen backbone, train only the head) with **full fine-tuning**,
3. evaluated the final model (accuracy, confusion matrix, per-class accuracy),
4. saved it in the **shared contract** format that Notebooks 02/03 and the Day-3 Flask API consume:
   `../models/mobilenetv2_cifar10_fp32.pth` and `../models/labels.json`.

| Item | Value |
|---|---|
| Dataset | CIFAR-10 (32×32 RGB, 10 classes) — we use a **10 000-image training subset** |
| Input size | upscaled to **96×96** (compromise between accuracy and CPU speed) |
| Runtime (CPU) | ~12–15 min with `QUICK_RUN = False`, ~2 min with `QUICK_RUN = True` |

> **Trainer tip:** start the full run *before* the lecture part on transfer learning, then discuss the
> theory while it trains.

## 0. Setup

`QUICK_RUN = True` shrinks everything (fewer images, 1 epoch) so you can check the notebook runs
end-to-end in ~2 minutes. For the real model use `QUICK_RUN = False`.

In quick mode the notebook will **not overwrite** an existing model in `../models/`.

In [ ]:
QUICK_RUN = False          # True -> smoke test (tiny subset, 1 epoch)

import json, time, random, copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("..") / "data"
MODELS_DIR = Path("..") / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 96
TRAIN_SUBSET = 1000 if QUICK_RUN else 10_000
TEST_SUBSET = 500 if QUICK_RUN else 2_000
FE_EPOCHS = 1                          # feature-extraction experiment
FT_EPOCHS = 1 if QUICK_RUN else 3      # full fine-tuning (our final model)
BATCH_SIZE = 64

LABELS = ["airplane", "automobile", "bird", "cat", "deer",
          "dog", "frog", "horse", "ship", "truck"]
MEAN = [0.485, 0.456, 0.406]   # ImageNet channel means
STD = [0.229, 0.224, 0.225]    # ImageNet channel stds

print(f"torch {torch.__version__} | CPU threads: {torch.get_num_threads()} | QUICK_RUN={QUICK_RUN}")

The paths are **relative to this notebook** (`Day2_Transfer_Learning_and_Optimization/`), so data lands in
the course-wide `data/` folder and models in the course-wide `models/` folder that Day 3 reads from.

## 1. Data pipeline — and why the numbers `0.485, 0.456, 0.406` matter

The pretrained backbone learned its filters on images normalised with the **ImageNet mean and standard
deviation**. If we fed it differently scaled pixels, every first-layer activation would be shifted and the
pretrained features would be much less useful. **Rule: always reuse the preprocessing the backbone was
trained with.**

* **Resize 32 → 96:** MobileNetV2 downsamples by 32×. A 32×32 input would end as a single 1×1 feature map
  after very few layers; at 96×96 we keep a 3×3 map at the end and much more useful intermediate detail.
  224×224 would be more accurate but ~5× slower on a CPU.
* **Augmentation (training only):** random crop (with reflection padding) + horizontal flip. They create
  "new" plausible images for free and reduce over-fitting. A cat flipped left-right is still a cat — but we
  would **not** flip digits or text!
* **Test transform = inference contract** for Day 3: `Resize((96,96)) → ToTensor → Normalize(mean, std)`.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 12, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tf = transforms.Compose([          # == Day-3 inference preprocessing
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_full = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_tf)
test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)
assert train_full.classes == LABELS

def fixed_subset(n_total, n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.randperm(n_total, generator=g)[:n].tolist()

train_ds = Subset(train_full, fixed_subset(len(train_full), TRAIN_SUBSET, SEED))
test_ds = Subset(test_full, fixed_subset(len(test_full), TEST_SUBSET, SEED))

# num_workers=0: the safest choice inside notebooks on Windows (no multiprocessing pickling issues)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=200, shuffle=False, num_workers=0)

counts = np.bincount([train_full.targets[i] for i in train_ds.indices], minlength=10)
print(f"train: {len(train_ds)} images | test: {len(test_ds)} images")
print("images per class (train subset):", dict(zip(LABELS, counts.tolist())))

A random subset of 10 000 images is still roughly **class-balanced** (~1 000 per class), so plain accuracy is
a fair metric. The test subset is fixed by the seed, so every notebook (and the CLI script) evaluates on
**exactly the same 2 000 test images** — results are comparable across the day.

Let's look at a few augmented training images (we undo the normalisation for display).

In [ ]:
def denorm(x):
    return (x * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)).clamp(0, 1)

xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lab in zip(axes.flat, xb[:16], yb[:16]):
    ax.imshow(denorm(img).permute(1, 2, 0)); ax.set_title(LABELS[lab], fontsize=9); ax.axis("off")
plt.suptitle(f"Augmented training batch (upscaled to {IMG_SIZE}×{IMG_SIZE}) — note the blur: CIFAR is only 32×32")
plt.tight_layout(); plt.show()
print("batch tensor:", tuple(xb.shape), xb.dtype, "| value range after Normalize:",
      f"[{xb.min():.2f}, {xb.max():.2f}]")

Note the tensor layout **NCHW** (batch, channels, height, width) and that normalised values are roughly in
`[-2, 2.6]`, not `[0, 1]`. This is exactly what the ONNX model will expect on Day 3.

## 2. The pretrained model and its new head

MobileNetV2 = `features` (convolutional backbone, 19 blocks of *inverted residuals* with depthwise-separable
convolutions) + `classifier` = `Dropout(0.2) → Linear(1280, 1000)`.

We keep everything and replace **only** `classifier[1]` with `Linear(1280, 10)` — that layer is randomly
initialised, everything else starts from the ImageNet weights.

In [ ]:
def build_model():
    model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)   # downloads ~14 MB once
    model.classifier[1] = nn.Linear(model.last_channel, len(LABELS))   # 1280 -> 10 (Dropout kept)
    return model

model = build_model()
print(model.classifier)
n_total = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.classifier.parameters())
print(f"total parameters: {n_total/1e6:.2f} M | new head: {n_head:,} ({100*n_head/n_total:.2f} %)")
print("number of blocks in model.features:", len(model.features))

Only ~0.6 % of the parameters are new. The remaining 2.2 M parameters already know how to detect edges,
textures, object parts… That prior knowledge is what makes training on just 10 000 small images work.

### Helper functions: one training epoch and evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()                                   # BatchNorm uses running stats, Dropout off
    correct = total = 0
    preds, targets = [], []
    for x, y in loader:
        p = model(x).argmax(1)
        correct += (p == y).sum().item(); total += y.numel()
        preds.append(p); targets.append(y)
    return correct / total, torch.cat(preds), torch.cat(targets)


def fit(model, loader, epochs, lr, params=None, log_every=40):
    params = [p for p in model.parameters() if p.requires_grad] if params is None else params
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=epochs * len(loader), pct_start=0.15)
    history = []
    for epoch in range(epochs):
        model.train()
        t0, run_loss, seen, correct = time.time(), 0.0, 0, 0
        for i, (x, y) in enumerate(loader):
            optimizer.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step(); scheduler.step()
            run_loss += loss.item() * y.size(0); seen += y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            history.append(loss.item())
            if (i + 1) % log_every == 0:
                print(f"  epoch {epoch+1} step {i+1}/{len(loader)}  loss {run_loss/seen:.3f}  "
                      f"train-acc {correct/seen:.3f}")
        test_acc, _, _ = evaluate(model, test_loader)
        print(f"epoch {epoch+1}/{epochs}: loss {run_loss/seen:.3f} | train-acc {correct/seen:.3f} | "
              f"test-acc {test_acc:.4f} | {time.time()-t0:.0f}s")
    return history

Design choices worth mentioning:

* **AdamW + One-Cycle LR schedule:** short warm-up, then decay — converges fast in few epochs, which is what
  we need on a CPU.
* **Label smoothing 0.1:** a mild regulariser; targets become 0.91/0.01 instead of 1/0.
* `model.train()` vs `model.eval()` matters for **BatchNorm** and **Dropout** — forgetting `eval()` at test
  time is one of the most common bugs.

## 3. Experiment A — Feature extraction (frozen backbone)

We set `requires_grad = False` for every backbone parameter. Only the head is trained. Two consequences:

1. The optimiser only sees 12 810 parameters.
2. No gradients are computed for the backbone weights → the backward pass is cheaper.

⚠️ Subtle point: freezing weights does **not** freeze BatchNorm *running statistics*. In `train()` mode BN
still updates its running mean/var. For pure feature extraction we therefore keep the backbone in `eval()`
mode during training.

In [ ]:
torch.manual_seed(SEED)
fe_model = build_model()
for p in fe_model.features.parameters():
    p.requires_grad = False

n_trainable = sum(p.numel() for p in fe_model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_trainable:,}")

# keep BatchNorm statistics of the frozen backbone fixed: override train() for this experiment
class FrozenBackbone(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def train(self, mode=True):
        super().train(mode); self.m.features.eval(); return self
    def forward(self, x):
        return self.m(x)

fe_wrapped = FrozenBackbone(fe_model)
t0 = time.time()
fe_hist = fit(fe_wrapped, train_loader, epochs=FE_EPOCHS, lr=3e-3)
fe_time = time.time() - t0
fe_acc, _, _ = evaluate(fe_wrapped, test_loader)
print(f"\nFeature extraction: test accuracy {fe_acc:.4f} in {fe_time:.0f}s")

A linear classifier on frozen ImageNet features already works reasonably well — but ImageNet features at
96 px are not perfectly suited to tiny, blurry 32 px CIFAR images. Let the whole network adapt.

## 4. Experiment B — Full fine-tuning (our final model)

All 2.2 M parameters are trained, with a **smaller learning rate** than for the head alone (1e-3 vs 3e-3):
we want to *nudge* the pretrained weights, not destroy them ("catastrophic forgetting").

In [ ]:
torch.manual_seed(SEED)
model = build_model()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_trainable:,}")
t0 = time.time()
ft_hist = fit(model, train_loader, epochs=FT_EPOCHS, lr=1e-3)
ft_time = time.time() - t0
ft_acc, preds, targets = evaluate(model, test_loader)
print(f"\nFull fine-tuning: test accuracy {ft_acc:.4f} in {ft_time/60:.1f} min")

## 5. Comparison

In [ ]:
import pandas as pd
cmp = pd.DataFrame([
    {"strategy": "feature extraction (frozen backbone)", "epochs": FE_EPOCHS,
     "trainable params": 12_810, "train time [s]": round(fe_time), "test accuracy": round(fe_acc, 4)},
    {"strategy": "full fine-tuning", "epochs": FT_EPOCHS,
     "trainable params": sum(p.numel() for p in model.parameters()),
     "train time [s]": round(ft_time), "test accuracy": round(ft_acc, 4)},
])
display(cmp)

fig, ax = plt.subplots(figsize=(9, 3.5))
k = 10
smooth = lambda h: np.convolve(h, np.ones(k) / k, mode="valid")
ax.plot(smooth(fe_hist), label="feature extraction")
ax.plot(smooth(ft_hist), label="full fine-tuning")
ax.set_xlabel("training step"); ax.set_ylabel("loss (moving avg)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Training loss"); plt.show()

**How to read this:**

* Feature extraction is cheaper per step and needs almost no tuning — a great *baseline* and the right
  choice when you have very little data (hundreds of images) or when the new domain is close to ImageNet.
* Full fine-tuning is slower but adapts all features to the new domain and wins clearly here.
* A popular middle ground is to **freeze the early layers** (generic edges/textures) and fine-tune the later
  ones (task-specific) → Exercise 2 in the student lab.

## 6. Where does the model still make mistakes?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(targets, preds, labels=range(10))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix — test subset (acc {ft_acc:.3f})"); plt.tight_layout(); plt.show()

per_class = cm.diagonal() / cm.sum(1)
for lab, acc in sorted(zip(LABELS, per_class), key=lambda t: t[1]):
    print(f"{lab:>10s}: {acc:.3f}")

Typically the **animal classes** are hardest — above all **cat** (most often confused with dog), followed by
bird and dog — while vehicles, frogs and ships are easy. Semantically similar classes at very low resolution
are hard for humans too. This is a good point to discuss *what* accuracy hides: a single number never tells
you which errors your users will see.

A few test images with predictions (red = wrong):

In [ ]:
x_show = torch.stack([test_ds[i][0] for i in range(16)])
y_show = [test_ds[i][1] for i in range(16)]
model.eval()
with torch.no_grad():
    prob = model(x_show).softmax(1)
fig, axes = plt.subplots(2, 8, figsize=(14, 4.2))
for ax, img, y, p in zip(axes.flat, x_show, y_show, prob):
    ax.imshow(denorm(img).permute(1, 2, 0)); ax.axis("off")
    c = p.argmax().item()
    ax.set_title(f"{LABELS[c]} {p[c]:.2f}", fontsize=9, color="green" if c == y else "red")
plt.tight_layout(); plt.show()

Note that the model returns **logits**; we applied `softmax` here only for display. The ONNX model we
serve on Day 3 also returns raw logits and the API applies softmax.

## 7. Save the model — the shared contract

* `mobilenetv2_cifar10_fp32.pth` — the **state_dict** (weights only, no pickled code — the robust way).
  To load it, rebuild the architecture exactly (`mobilenet_v2()` + `classifier[1] = Linear(1280, 10)`).
* `labels.json` — class names in index order.

In [ ]:
pth_path = MODELS_DIR / "mobilenetv2_cifar10_fp32.pth"
if QUICK_RUN and pth_path.exists():
    print(f"QUICK_RUN: not overwriting existing {pth_path} (smoke-test model is not good enough)")
else:
    torch.save(model.state_dict(), pth_path)
    (MODELS_DIR / "labels.json").write_text(json.dumps(LABELS, indent=2))
    print(f"saved {pth_path}  ({pth_path.stat().st_size/1e6:.2f} MB)")
    print("saved", MODELS_DIR / "labels.json")

### Sanity check: reload from disk and re-evaluate

Always verify a saved artefact by loading it into a *fresh* object — this catches missing keys, wrong
head sizes, or forgetting `eval()`.

In [ ]:
reloaded = mobilenet_v2(weights=None)
reloaded.classifier[1] = nn.Linear(1280, 10)
reloaded.load_state_dict(torch.load(pth_path, map_location="cpu", weights_only=True))
acc_reloaded, _, _ = evaluate(reloaded, test_loader)
print(f"accuracy of the model on disk: {acc_reloaded:.4f}")
if not QUICK_RUN:
    assert abs(acc_reloaded - ft_acc) < 1e-6, "reloaded model differs from trained model!"

## Summary

* Transfer learning = reuse a pretrained backbone + new task-specific head.
* **Feature extraction** (frozen backbone) is fast and a strong baseline; **fine-tuning** adapts all layers
  and gives the best accuracy when you have enough data.
* Reuse the backbone's **preprocessing** (ImageNet mean/std), augment only the training data.
* We saved an FP32 model of ~9 MB. **Next (Notebook 02):** how small and fast can we make it —
  and what does it cost in accuracy?